In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import csv
import re
import time

# Setup Chrome in Headless Mode
def setup_driver():
    options = Options()
    options.add_argument("--headless")  # Run without opening a window
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920x1080")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

# Extract raw event text data from page source
def extract_event_text(page_source, page_number):
    """Extracts the raw text of the event JSON from the rendered page source."""
    
    # Save full HTML for debugging
    with open(f"debug_raw_page_{page_number}.html", "w", encoding="utf-8") as f:
        f.write(page_source)

    print(f"Saved raw HTML for page {page_number} for debugging.")

    # Regex pattern to capture everything between 'var events =' and the next semicolon
    match = re.search(r'var\s+events\s*=\s*(\[.*?\]);', page_source, re.DOTALL)
    
    if match:
        events_text = match.group(1).strip()
        
        # Save raw extracted JSON text for debugging
        with open(f"debug_extracted_events_{page_number}.txt", "w", encoding="utf-8") as f:
            f.write(events_text)
        
        print(f"Extracted event text successfully for page {page_number}")

        return events_text  # Returning as raw text for now
    else:
        print(f"No events JSON found on page {page_number}.")
        return None

# Get the number of pages from pagination
def get_total_pages(driver, base_url):
    """Extracts the total number of pages from the pagination UI."""
    driver.get(base_url + "1")
    time.sleep(5)  # Allow JavaScript to load fully

    pagination_links = driver.find_elements("css selector", "ul.pagination li a")
    page_numbers = [int(link.text.strip()) for link in pagination_links if link.text.strip().isdigit()]
    
    return max(page_numbers) if page_numbers else 1

# Scrape all event data from pagination
def scrape_events():
    """Scrapes all event text data by iterating over paginated pages."""
    base_url = "https://adcc.smoothcomp.com/en/federation/176/events/past?page="
    driver = setup_driver()

    total_pages = get_total_pages(driver, base_url)
    all_events_raw = []  # Store raw extracted event text

    for page in range(1, total_pages + 1):
        print(f"Fetching page {page} of {total_pages}...")
        driver.get(base_url + str(page))
        time.sleep(5)  # Ensure JavaScript loads

        events_text = extract_event_text(driver.page_source, page)
        if events_text:
            all_events_raw.append(events_text)

    driver.quit()

    # Save extracted raw event text to a single file
    with open("all_events_raw.txt", "w", encoding="utf-8") as f:
        for event_text in all_events_raw:
            f.write(event_text + "\n\n")

    print(f"Successfully saved raw events text to all_events_raw.txt")

    return all_events_raw

if __name__ == "__main__":
    events = scrape_events()
    print(f"Extracted event data saved to all_events_raw.txt")


Fetching page 1 of 2...
Saved raw HTML for page 1 for debugging.
No events JSON found on page 1.
Fetching page 2 of 2...
Saved raw HTML for page 2 for debugging.
No events JSON found on page 2.
Successfully saved raw events text to all_events_raw.txt
Extracted event data saved to all_events_raw.txt
